In [ ]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

In [ ]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib
import streamlit as st
import plotly.graph_objects as go
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu

APP_NAME = "Infosys Franchise Analytics & Management"
APP_SYMBOL = "🏢"
DISPLAY_NAME = "Bhavya Sree"

st.set_page_config(page_title=APP_NAME, page_icon=APP_SYMBOL, layout="wide")

JWT_SECRET = "super-secret-infosys-key-2026"
SENDER_EMAIL = "bhavyasreegujjula@gmail.com"
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
OTP_EXPIRY_MINUTES = 5

ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"

COLORS = {
    "bg_main": "#f8fafc",
    "bg_sidebar": "#e3f6f5",
    "bg_card": "#ffffff",
    "bg_card_alt": "#bae8e8",
    "text_main": "#334155",
    "text_heading": "#272343",
    "text_muted": "#64748b",
    "accent": "#ffd803",
    "accent_hover": "#e6c300",
    "accent_text": "#272343",
    "border": "#272343",
    "border_light": "#bae8e8"
}

st.markdown(f"""
<style>
html, body, .stApp {{
    background: {COLORS["bg_main"]} !important;
    color: {COLORS["text_main"]} !important;
}}
.block-container {{
    padding: 2rem 2.5rem !important;
    max-width: 1200px;
}}
h1, h2, h3, h4 {{
    color: {COLORS["text_heading"]} !important;
}}
div[data-testid="stButton"] button {{
    background-color: {COLORS["accent"]} !important;
    color: {COLORS["accent_text"]} !important;
    border: 2px solid {COLORS["border"]} !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    height: 48px !important;
    box-shadow: 4px 4px 0px {COLORS["border"]} !important;
    width: 100%;
}}
div[data-testid="stButton"] button:hover {{
    background-color: {COLORS["accent_hover"]} !important;
}}
div[data-baseweb="input"], div[data-baseweb="select"] {{
    background-color: {COLORS["bg_card"]} !important;
    border: 2px solid {COLORS["border"]} !important;
    border-radius: 10px !important;
}}
section[data-testid="stSidebar"] {{
    background: {COLORS["bg_sidebar"]} !important;
    border-right: 2px solid {COLORS["border"]} !important;
}}
</style>
""", unsafe_allow_html=True)

def get_db():
    return sqlite3.connect("infosys_portal.db", check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    return bcrypt.checkpw(t.encode(), h.encode()) if h else False

def validate_email(email):
    if not email or "@" not in email or "." not in email:
        return False
    try:
        before_at, after_at = email.split("@", 1)
        domain_name, domain_ext = after_at.rsplit(".", 1)
    except ValueError:
        return False

    return (
        len(before_at) >= 2
        and len(domain_name) >= 2
        and len(domain_ext) >= 2
        and before_at.isalpha()
        and domain_name.isalpha()
        and domain_ext.isalpha()
    )

def validate_password(password):
    if len(password) < 8:
        return False, "Password must be at least 8 characters long."
    if not any(ch.islower() for ch in password):
        return False, "Password must include at least one lowercase letter."
    if not any(ch.isupper() for ch in password):
        return False, "Password must include at least one uppercase letter."
    if not any(ch.isdigit() for ch in password):
        return False, "Password must include at least one number."
    special_chars = "!@#$%^&*()-_=+[]{};:'\",.<>?/\\|`~"
    if not any(ch in special_chars for ch in password):
        return False, "Password must include at least one special character."
    return True, ""

def is_same_old_password(email, new_password):
    with get_db() as c:
        r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
    return bool(r and check_txt(new_password, r[0]))

def get_registered_users():
    with get_db() as c:
        return c.execute(
            "SELECT username, email FROM users ORDER BY username"
        ).fetchall()

with get_db() as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            security_question TEXT,
            security_answer_hash TEXT
        )
    """)

def make_jwt(identity, role="user"):
    return jwt.encode(
        {
            "sub": identity,
            "role": role,
            "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)
        },
        JWT_SECRET,
        algorithm="HS256"
    )

def verify_jwt(token):
    try:
        return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None

def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {
        "sub": email,
        "otp_hash": hash_txt(otp),
        "type": "password_reset_otp",
        "iat": datetime.datetime.utcnow(),
        "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email:
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid OTP."
    except jwt.ExpiredSignatureError:
        return False, "OTP expired. Please request a new one."
    except Exception:
        return False, "Invalid verification token."

def send_professional_email(to_email, otp, app_pass):
    msg = MIMEMultipart("alternative")
    msg["From"] = f"Infosys Support <{SENDER_EMAIL}>"
    msg["To"] = to_email
    msg["Subject"] = f"{APP_NAME} - Verification Code"
    msg["Date"] = formatdate(localtime=True)
    msg["Message-ID"] = make_msgid()
    msg["Reply-To"] = SENDER_EMAIL

    text_body = f"Your verification code for {APP_NAME} is: {otp}\nThis code expires in {OTP_EXPIRY_MINUTES} minutes."

    html_body = f"""
    <html>
    <body style="font-family: Arial; background:#f9fcfc; padding:20px;">
        <div style="max-width:500px; margin:auto; background:white; border:2px solid #272343; border-radius:12px; padding:30px; text-align:center;">
            <h2>{APP_NAME} Verification</h2>
            <p>Your password reset code is:</p>
            <div style="background:#ffd803; font-size:28px; font-weight:bold; letter-spacing:5px; padding:15px; border:2px solid #272343; border-radius:8px;">
                {otp}
            </div>
            <p>This code expires in {OTP_EXPIRY_MINUTES} minutes.</p>
        </div>
    </body>
    </html>
    """

    msg.attach(MIMEText(text_body, "plain"))
    msg.attach(MIMEText(html_body, "html"))

    try:
        s = smtplib.SMTP("smtp.gmail.com", 587)
        s.starttls()
        s.login(SENDER_EMAIL, app_pass)
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent."
    except Exception as e:
        return False, str(e)

for k, v in [
    ("token", None),
    ("page", "Login"),
    ("reset_email", None),
    ("reset_mode", None),
    ("otp_stage", "send"),
    ("otp_token", ""),
    ("sq_p", "")
]:
    if k not in st.session_state:
        st.session_state[k] = v

def navigate(page):
    st.session_state.page = page
    st.rerun()

def auth_header(title, sub="Franchise Analytics & Management Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:42px;margin-bottom:8px;">{APP_SYMBOL}</div>
        <h1 style="font-size:2rem;margin:0;">{APP_NAME}</h1>
        <p style="color:{COLORS["text_muted"]};font-size:14px;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;">
        <b>{title}</b>
    </div>
    """, unsafe_allow_html=True)

def show_health_graph():
    fig = go.Figure(
        go.Indicator(
            mode="gauge+number",
            value=92,
            number={"font": {"size": 76, "color": "#272343"}},
            title={"text": "System Health Index", "font": {"size": 16, "color": "#272343"}},
            gauge={
                "shape": "angular",
                "axis": {
                    "range": [0, 100],
                    "tickmode": "array",
                    "tickvals": [0, 20, 40, 60, 80, 100],
                    "ticktext": ["0", "20", "40", "60", "80", "100"],
                    "tickwidth": 1,
                    "tickcolor": "#272343",
                    "tickfont": {"size": 12, "color": "#272343"}
                },
                "bar": {"color": "#ffd803", "thickness": 0.28},
                "bgcolor": "#bae8e8",
                "borderwidth": 1,
                "bordercolor": "#272343",
                "steps": [{"range": [0, 100], "color": "#bae8e8"}]
            }
        )
    )
    fig.update_layout(
        height=390,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=20, r=20, t=55, b=10),
        font={"family": "Inter", "color": "#272343"}
    )
    _, graph_col, _ = st.columns([1, 1.55, 1])
    with graph_col:
        st.plotly_chart(fig, use_container_width=True)

if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])

    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            login_id = st.text_input("Username / Email").strip()
            pwd = st.text_input("Password", type="password")

            col1, col2, col3 = st.columns([1, 1.15, 1.3])

            if col1.button("Sign In", use_container_width=True):
                if not login_id or not pwd:
                    st.error("Please fill all required fields.")
                elif login_id == ADMIN_USERNAME and pwd == ADMIN_PASSWORD:
                    st.session_state.token = make_jwt(ADMIN_USERNAME, role="admin")
                    navigate("Dashboard")
                elif "@" in login_id and not validate_email(login_id.lower()):
                    st.error("Please enter a valid email address.")
                else:
                    with get_db() as c:
                        r = c.execute(
                            "SELECT email, password_hash FROM users WHERE email=? OR username=?",
                            (login_id.lower(), login_id)
                        ).fetchone()

                    if r and check_txt(pwd, r[1]):
                        st.session_state.token = make_jwt(r[0], role="user")
                        navigate("Dashboard")
                    else:
                        st.error("Invalid credentials.")

            if col2.button("Create Account", use_container_width=True):
                navigate("Signup")

            if col3.button("Forgot Password", use_container_width=True):
                navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account")
            uname = st.text_input("Username")
            email = st.text_input("Email address").lower().strip()
            pwd = st.text_input("Password", type="password")
            confirm_pwd = st.text_input("Confirm password", type="password")
            sq = st.selectbox("Security Question", [
                "What is your pet name?",
                "What is your mother's maiden name?",
                "What is your favourite city?"
            ])
            sa = st.text_input("Security answer")

            if st.button("Create Account", use_container_width=True):
                valid_password, password_msg = validate_password(pwd)

                if not uname or not email or not pwd or not confirm_pwd or not sq or not sa:
                    st.error("Please fill all required fields.")
                elif uname == ADMIN_USERNAME:
                    st.error("This username is reserved. Please choose another username.")
                elif not validate_email(email):
                    st.error("Please enter a valid email address.")
                elif not valid_password:
                    st.error(password_msg)
                elif pwd != confirm_pwd:
                    st.error("Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute(
                                "INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                                (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip()))
                            )

                        st.success("Account created successfully. Please login.")
                        time.sleep(1)
                        navigate("Login")

                    except sqlite3.IntegrityError:
                        st.error("Email or username already registered.")

            if st.button("Back to Sign In", use_container_width=True):
                navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password")

            if not st.session_state.reset_email:
                email = st.text_input("Registered email address").lower().strip()
                col1, col2 = st.columns(2)

                if col1.button("Via Security Question", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    elif not validate_email(email):
                        st.error("Please enter a valid email address.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            st.session_state.reset_email = email
                            st.session_state.sq_p = r[0]
                            st.session_state.reset_mode = "sq"
                            st.rerun()
                        else:
                            st.error("Email not found.")

                if col2.button("Via OTP", use_container_width=True):
                    if not email:
                        st.error("Please enter your registered email address.")
                    elif not validate_email(email):
                        st.error("Please enter a valid email address.")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT id FROM users WHERE email=?", (email,)).fetchone()
                        if r:
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.session_state.otp_stage = "send"
                            st.rerun()
                        else:
                            st.error("Email not found.")

            else:
                if st.session_state.reset_mode == "sq":
                    st.info(f"Security Question: {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password", type="password")
                    cpw = st.text_input("Confirm new password", type="password")

                    if st.button("Reset Password", use_container_width=True):
                        valid_password, password_msg = validate_password(npw)

                        with get_db() as c:
                            r = c.execute(
                                "SELECT security_answer_hash FROM users WHERE email=?",
                                (st.session_state.reset_email,)
                            ).fetchone()

                        if not ans or not npw or not cpw:
                            st.error("Please fill all required fields.")
                        elif not valid_password:
                            st.error(password_msg)
                        elif npw != cpw:
                            st.error("Passwords do not match.")
                        elif is_same_old_password(st.session_state.reset_email, npw):
                            st.error("You cannot reuse your old password. Please choose another password.")
                        elif r and check_txt(ans, r[0]):
                            with get_db() as c:
                                c.execute(
                                    "UPDATE users SET password_hash=? WHERE email=?",
                                    (hash_txt(npw), st.session_state.reset_email)
                                )
                            st.success("Password updated. Please login.")
                            time.sleep(1)
                            st.session_state.reset_email = None
                            st.session_state.reset_mode = None
                            navigate("Login")
                        else:
                            st.error("Incorrect answer.")

                elif st.session_state.reset_mode == "otp":
                    if st.session_state.otp_stage == "send":
                        st.info(f"Send OTP to {st.session_state.reset_email}")

                        if st.button("Send Verification Code", use_container_width=True):
                            if not EMAIL_PASSWORD:
                                st.error("Gmail App Password not found. Add EMAIL_PASSWORD to Colab Secrets.")
                            else:
                                otp = generate_otp()
                                ok, msg = send_professional_email(st.session_state.reset_email, otp, EMAIL_PASSWORD)

                                if ok:
                                    st.session_state.otp_token = make_otp_token(st.session_state.reset_email, otp)
                                    st.session_state.otp_stage = "verify"
                                    st.success("OTP sent. Check your inbox.")
                                    time.sleep(1)
                                    st.rerun()
                                else:
                                    st.error(msg)

                    elif st.session_state.otp_stage == "verify":
                        otp_input = st.text_input("Enter 6-digit OTP", max_chars=6)

                        if st.button("Verify Code", use_container_width=True):
                            if not otp_input:
                                st.error("Please enter the OTP.")
                            else:
                                ok, msg = verify_otp_token(
                                    st.session_state.otp_token,
                                    otp_input,
                                    st.session_state.reset_email
                                )

                                if ok:
                                    st.session_state.otp_stage = "reset"
                                    st.rerun()
                                else:
                                    st.error(msg)

                    elif st.session_state.otp_stage == "reset":
                        npw = st.text_input("New password", type="password")
                        cpw = st.text_input("Confirm new password", type="password")

                        if st.button("Update Password", use_container_width=True):
                            valid_password, password_msg = validate_password(npw)

                            if not npw or not cpw:
                                st.error("Please fill all required fields.")
                            elif not valid_password:
                                st.error(password_msg)
                            elif npw != cpw:
                                st.error("Passwords do not match.")
                            elif is_same_old_password(st.session_state.reset_email, npw):
                                st.error("You cannot reuse your old password. Please choose another password.")
                            else:
                                with get_db() as c:
                                    c.execute(
                                        "UPDATE users SET password_hash=? WHERE email=?",
                                        (hash_txt(npw), st.session_state.reset_email)
                                    )
                                st.success("Password updated. Please login.")
                                time.sleep(1)
                                st.session_state.reset_email = None
                                st.session_state.reset_mode = None
                                st.session_state.otp_stage = "send"
                                st.session_state.otp_token = ""
                                navigate("Login")

            if st.button("Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_stage = "send"
                st.session_state.otp_token = ""
                navigate("Login")

else:
    payload = verify_jwt(st.session_state.token)

    if not payload:
        st.session_state.token = None
        navigate("Login")

    role = payload.get("role", "user")

    if role == "admin":
        with st.sidebar:
            st.markdown(f"""<div style="padding:44px 8px 30px;text-align:center;">
<div style="font-size:34px;color:{COLORS['accent']};">🛡️</div>
<div style="font-weight:800;color:{COLORS['text_heading']};font-size:18px;margin-top:8px;">Admin Panel</div>
<div style="font-size:12px;color:{COLORS['text_muted']};margin-top:4px;">User Management</div>
</div>
<hr style="border:0;border-top:1px solid {COLORS['border_light']};margin:0 12px 28px;">""", unsafe_allow_html=True)

            menu = option_menu(
                None,
                ["Admin Dashboard", "Logout"],
                icons=["shield-lock", "box-arrow-right"],
                default_index=0,
                styles={
                    "container": {"background-color": COLORS["bg_sidebar"], "padding": "0 14px"},
                    "nav-link": {
                        "font-size": "17px",
                        "color": COLORS["text_heading"],
                        "font-weight": "500",
                        "margin": "8px 0",
                        "padding": "12px 18px",
                        "border-radius": "8px"
                    },
                    "nav-link-selected": {
                        "background-color": COLORS["accent"],
                        "color": COLORS["accent_text"],
                        "font-weight": "800"
                    }
                }
            )

            if menu == "Logout":
                st.session_state.token = None
                navigate("Login")

        users = get_registered_users()
        total_users = len(users)

        st.markdown(f"""<div style="background:{COLORS['text_heading']};border-radius:16px;padding:34px 38px;margin:8px 0 26px;display:flex;justify-content:space-between;align-items:center;">
<div>
<div style="font-size:46px;color:{COLORS['accent']};line-height:1;">🛡️</div>
<h1 style="color:{COLORS['accent']} !important;margin:14px 0 0;font-size:28px;">Admin Dashboard</h1>
<p style="color:{COLORS['bg_card_alt']};margin:8px 0 0;">Registered Users</p>
</div>
<div style="background:{COLORS['accent']};color:{COLORS['accent_text']};padding:10px 22px;border-radius:30px;font-size:14px;font-weight:800;">
Total Users: {total_users}
</div>
</div>""", unsafe_allow_html=True)

        if users:
            st.markdown("### Registered Users")

            for idx, (username, email) in enumerate(users, start=1):
                st.markdown(f"""<div style="background:#ffffff;border:2px solid {COLORS['border']};border-radius:12px;padding:18px 22px;margin-bottom:12px;box-shadow:5px 5px 0px {COLORS['border_light']};display:flex;justify-content:space-between;align-items:center;">
<div>
<div style="font-size:13px;color:{COLORS['text_muted']};font-weight:700;">USER {idx}</div>
<div style="font-size:20px;color:{COLORS['text_heading']};font-weight:900;margin-top:4px;">{username}</div>
</div>
<div style="font-size:15px;color:{COLORS['text_main']};font-weight:700;">{email}</div>
</div>""", unsafe_allow_html=True)
        else:
            st.info("No registered users found. Create a normal user from the Signup page, then login as admin again.")

        st.stop()

    with st.sidebar:
        st.markdown(f"""<div style="padding:44px 8px 30px;text-align:center;">
<div style="font-size:34px;color:{COLORS['accent']};">⚡</div>
<div style="font-weight:800;color:{COLORS['text_heading']};font-size:18px;margin-top:8px;">Infosys Portal</div>
<div style="font-size:12px;color:{COLORS['text_muted']};margin-top:4px;">Intelligent Analytics</div>
</div>
<hr style="border:0;border-top:1px solid {COLORS['border_light']};margin:0 12px 28px;">""", unsafe_allow_html=True)

        menu = option_menu(
            None,
            ["Dashboard", "Analytics", "Reports", "Logout"],
            icons=["house", "graph-up", "file-text", "box-arrow-right"],
            default_index=0,
            styles={
                "container": {"background-color": COLORS["bg_sidebar"], "padding": "0 14px"},
                "nav-link": {
                    "font-size": "17px",
                    "color": COLORS["text_heading"],
                    "font-weight": "500",
                    "margin": "8px 0",
                    "padding": "12px 18px",
                    "border-radius": "8px"
                },
                "nav-link-selected": {
                    "background-color": COLORS["accent"],
                    "color": COLORS["accent_text"],
                    "font-weight": "800"
                }
            }
        )

        if menu == "Logout":
            st.session_state.token = None
            navigate("Login")

    st.markdown(f"""<div style="background:{COLORS['text_heading']};border-radius:16px;padding:34px 38px;margin:8px 0 26px;display:flex;justify-content:space-between;align-items:center;min-height:125px;">
<div>
<div style="font-size:48px;color:{COLORS['accent']};line-height:1;">⚡</div>
<div style="color:{COLORS['bg_card_alt']};font-size:14px;margin-top:18px;">Analytics Dashboard</div>
</div>
<div style="background:{COLORS['accent']};color:{COLORS['accent_text']};padding:10px 22px;border-radius:30px;font-size:14px;font-weight:800;margin-top:28px;">👤 {DISPLAY_NAME}</div>
</div>""", unsafe_allow_html=True)

    c1, c2, c3, c4 = st.columns(4)

    dashboard_cards = [
        (c1, "📄", "128", "Documents Indexed"),
        (c2, "🔍", "47", "Searches Today"),
        (c3, "📊", "98.4%", "Efficiency Score"),
        (c4, "🛡️", "Secured", "Security Status")
    ]

    for col, icon, value, label in dashboard_cards:
        col.markdown(f"""<div style="background:#ffffff;border:2px solid {COLORS['border']};border-radius:14px;min-height:145px;display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center;box-shadow:6px 6px 0px {COLORS['border_light']};">
<div style="font-size:34px;margin-bottom:14px;">{icon}</div>
<div style="font-size:28px;font-weight:900;color:{COLORS['text_heading']};line-height:1.1;">{value}</div>
<div style="font-size:13px;color:{COLORS['text_muted']};font-weight:600;margin-top:10px;">{label}</div>
</div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    show_health_graph()

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import time

ngrok.kill()
!pkill -f ngrok
!pkill -f streamlit

time.sleep(20)

print("Cleaned old sessions.")

Cleaned old sessions.


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

NGROK_TOKEN = userdata.get("NGROK_AUTHTOKEN")
EMAIL_PASSWORD = userdata.get("EMAIL_PASSWORD")

ngrok.set_auth_token(NGROK_TOKEN)
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

public_url = ngrok.connect(8501).public_url

print("Open this URL:", public_url)

Open this URL: https://payphone-tasty-enquirer.ngrok-free.dev
